# Dataset typicality: same generator or another one?

You fix a generator configuration up front and then ask of a dataset: **could it
be a world of this configuration, and if not, what gives it away?**
[`typicality`](../reference/typicality.md) answers with a calibrated p-value — the
share of the configuration's worlds of the same shape that are at least as
unusual as the dataset — and a per-descriptor profile.

This notebook fixes one configuration, **A**, and scores datasets that really
come from A against datasets from three other generators:

| Generator | What differs from A | Expected |
| --- | --- | --- |
| **A** | nothing: same configuration, other seeds | p spread over (0, 1] |
| **B1** | every input switches on and off (`trajectories="periodic_on_off"`) | weeks at exactly zero, which A never makes: always the smallest p |
| **B2** | inputs combine trends, seasonality, gates and pulses (`trajectories="composable"`) | mostly the smallest p |
| **C** | covariate levels range over ±20 instead of A's narrow range | often small p, not always: a subtle difference |

Every world has 52 weeks, 3 treatments and 2 covariates. Each dataset goes
through a CSV file with made-up column names, exactly as a real one would.

In [ ]:
%matplotlib inline

import dataclasses
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pymc_generator as pg

workdir = Path(tempfile.mkdtemp())
SHAPE = {"n_treatments": 3, "n_covariates": 2, "n_latent": 1, "n_time_steps": 52}

## 1 · Four generators, twenty datasets each

`n_cells=20, draws_per_cell=1`: every dataset comes from its own generation
cell, so no two share a graph or mechanisms. Generator A uses seed 1; the
reference built from A later uses another seed, so no dataset can be one of the
reference's own worlds.

In [ ]:
priors = {
    "A": pg.make_scm_prior(**SHAPE),
    "B1": pg.make_scm_prior(**SHAPE, trajectories="periodic_on_off"),
    "B2": pg.make_scm_prior(**SHAPE, trajectories="composable"),
    "C": pg.make_scm_prior(**SHAPE, rw_covariate_mean_range=(-20.0, 20.0)),
}
panels = {
    name: pg.sample_prior_predictive(
        dataclasses.replace(prior, n_cells=20, draws_per_cell=1, seed=seed)
    )
    for (name, prior), seed in zip(priors.items(), (1, 2, 3, 4))
}
print({name: corpus["treatment_raw"].shape for name, corpus in panels.items()})

## 2 · A dataset on disk

Take the first world of each panel, export it with `ObservedWorlds.to_frame`
(the audit-bundle layout: `week`, `treatment_C1…`, `covariate_Z1…`,
`outcome_Y`), rename the columns as a real export would name them, and write a
CSV file.

In [ ]:
NAMES = {
    "treatment_C1": "tv",
    "treatment_C2": "search",
    "treatment_C3": "social",
    "covariate_Z1": "price",
    "covariate_Z2": "temperature",
    "outcome_Y": "sales",
}


def world(corpus, i):
    # One corpus world as an ObservedWorlds (series only).
    keys = (
        "treatment_raw",
        "covariates",
        "outcome_raw",
        "treatment_active_mask",
        "covariate_active_mask",
    )
    return pg.ObservedWorlds(**{key: corpus[key][[i]] for key in keys})


frames = {
    name: world(corpus, 0).to_frame(0).rename(columns=NAMES) for name, corpus in panels.items()
}
for name, frame in frames.items():
    frame.to_csv(workdir / f"dataset_{name}.csv", index=False)
frames["B1"].head()

Reading it back only needs the roles: which columns are treatments, which are
covariates, which one is the outcome.

In [ ]:
datasets = {
    name: pg.ObservedWorlds.from_csv(
        workdir / f"dataset_{name}.csv",
        treatments=["tv", "search", "social"],
        covariates=["price", "temperature"],
        outcome="sales",
        time="week",
    )
    for name in frames
}
datasets["A"].treatment_raw.shape, datasets["A"].covariates.shape, datasets["A"].outcome_raw.shape

In [ ]:
# The CSV round trip keeps every value (generated worlds are float32).
for name, frame in frames.items():
    back = pd.read_csv(workdir / f"dataset_{name}.csv")
    assert (back.astype(frame.dtypes.to_dict()) == frame).all().all(), name

## 3 · The reference: configuration A at the dataset's shape

`matched_prior` conditions A on the dataset's shape (3 active treatments, 2
active covariates, 52 weeks) and sets the generation size: 200 cells, three
draws each. The default split sends 100 cells to calibration (one world each)
and 100 to the ruler (all three draws: extra draws are nearly free to generate
and a larger ruler can help against subtle differences), so the smallest possible
p is 1/101. The corpus is saved and loaded back to show
the "existing corpus" path: a stored corpus is used exactly like a freshly
generated one.

In [ ]:
reference_prior = pg.matched_prior(
    priors["A"], datasets["A"], n_cells=200, draws_per_cell=3, seed=101
)
pg.save_corpus(pg.sample_prior_predictive(reference_prior), workdir / "reference.npz")
reference_corpus = pg.load_corpus(workdir / "reference.npz")

LAGS = (1, 4)
reference = pg.world_descriptors(reference_corpus, source_id="reference A", lags=LAGS)
print(reference.n_worlds, "reference worlds,", len(set(reference.group_ids.tolist())), "cells")

In [ ]:
for column, value in (("n_treatments_active", 3), ("n_covariates_active", 2), ("n_time_steps", 52)):
    assert (reference.grouping(column) == value).all(), column

## 4 · Four datasets, one question each

In [ ]:
target = pg.WorldDescriptors.concat(
    [
        pg.world_descriptors(dataset, source_id=f"dataset {name}", lags=LAGS)
        for name, dataset in datasets.items()
    ]
)
result = pg.typicality(reference, target)
print(result.table())

In [ ]:
assert (result.to_frame()["status"] == "ok").all()
assert (result.n_calibration == 100).all()
b1 = list(datasets).index("B1")
assert result.violations[b1] > 0 and result.p_value[b1] == result.p_min[b1]

Each line is one dataset. `p_value` is `(1 + m) / (n + 1)`, where `m` of the
`n = 100` calibration worlds (A's worlds of this shape) score at least as
unusual as the dataset; `p_min` is the smallest value they allow. Read it as a rate, not a verdict: one in twenty of
A's own datasets gets p ≤ 0.05, so one small p for an A dataset is no
contradiction — one dataset is one draw.

In [ ]:
for name, p, p_min, n, v in zip(
    datasets, result.p_value, result.p_min, result.n_calibration, result.violations
):
    m = round(p * (n + 1)) - 1
    where = f"{m} of {n} calibration worlds at least as unusual" + (
        ", the floor" if p == p_min else ""
    )
    why = f"; {int(v)} support violations" if v else ""
    print(f"dataset {name}: p = {p:.3f} ({where}{why})")

What is not a matter of chance is B1. Its inputs have weeks at exactly zero:
every ruler world has a zero-week share of exactly 0, so those descriptors are
**support violations** (listed under the table), no calibration world has one,
and B1 gets `p_min` by construction. The profile says which descriptors gave it
away:

In [ ]:
COLUMNS = [
    "feature",
    "value",
    "percentile",
    "reference_low",
    "reference_median",
    "reference_high",
    "inside",
    "violation",
]


def standouts(row):
    profile = result.profile(row)
    return profile[profile["violation"] | ~profile["inside"]][COLUMNS]


standouts(b1)

C differs from A only in where covariate levels sit, so it rarely shows
anything outside A's support: what sets it apart, if anything, is
**distance**, and its profile points at the descriptors that moved:

In [ ]:
standouts(list(datasets).index("C"))

## 5 · Twenty datasets per generator

One dataset is one draw. Scoring each panel shows how the answer behaves
across draws. All 80 datasets are scored against the same reference, so their
p-values are not independent of each other.

In [ ]:
panel_rows = pg.WorldDescriptors.concat(
    [
        pg.world_descriptors(corpus, source_id=f"panel {name}", lags=LAGS)
        for name, corpus in panels.items()
    ]
)


def panel_summary(seed):
    scored = pg.typicality(reference, panel_rows, seed=seed)
    frame = scored.to_frame().assign(generator=lambda f: f["source_id"].str.removeprefix("panel "))
    return scored, frame.groupby("generator").agg(
        at_floor=("p_value", lambda p: int((p == scored.p_min[0]).sum())),
        p_at_most_005=("p_value", lambda p: int((p <= 0.05).sum())),
        median_p=("p_value", "median"),
        with_violations=("violations", lambda v: int((v > 0).sum())),
    )


panel_result, summary = panel_summary(seed=0)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
jitter = np.random.default_rng(0)
for x, name in enumerate(panels):
    p = panel_result.p_value[panel_result.source_ids == f"panel {name}"]
    ax.scatter(x + jitter.uniform(-0.15, 0.15, p.size), p, alpha=0.8)
floor = panel_result.p_min[0]
ax.axhline(floor, color="grey", lw=1, ls=":", label=f"smallest possible p (1/{round(1 / floor)})")
ax.axhline(0.05, color="grey", lw=1, ls="--", label="0.05")
ax.set_xticks(range(len(panels)), [f"generator {name}" for name in panels])
ax.set_yscale("log")
ax.set_ylabel("typicality p-value")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
ax.set_title("Twenty datasets per generator against reference A")
plt.show()

The guarantee — for datasets that really come from A, `P(p ≤ α) ≤ α` — is an
average over references and datasets: about one A dataset in twenty at
p ≤ 0.05, and anywhere from 0 to 3 out of twenty is ordinary. One reference
and one panel of twenty are each a single draw, so a run can land above or
below that. The split seed only reshuffles which of the same 200 cells
calibrate — it resamples neither the reference nor the panel — so re-scoring
under a few seeds shows how much the split alone moves the counts, and that it
never moves B1:

In [ ]:
by_seed = pd.concat(
    {seed: panel_summary(seed)[1][["at_floor", "p_at_most_005"]] for seed in range(5)},
    names=["seed"],
).unstack("generator")
by_seed

In [ ]:
for seed in range(5):
    scored, _ = panel_summary(seed)
    b1_rows = scored.source_ids == "panel B1"
    assert (scored.violations[b1_rows] > 0).all()
    assert (scored.p_value[b1_rows] == scored.p_min[b1_rows]).all()

How to read it:

- **A**: datasets from the configuration itself. Their p-values spread over
  the whole range; compare the count at p ≤ 0.05 with the one in twenty
  expected. A panel can hold a few unusual draws of A itself, and a reference
  can be on the strict side; neither breaks the guarantee, which is an average.
- **B1** sits at the floor under every split, by construction: on/off flighting
  makes zero weeks that A never produces.
- **B2**: gates (onsets, offsets, flighting) also put exact zero weeks into at
  least one input of most B2 datasets, so most B2 datasets break A's support
  like B1 and sit at the floor; the few without zero weeks are judged by
  distance.
- **C** differs from A only by degree and rarely breaks A's support, so its
  answer rests on distance: small p far more often than A, but not always. A
  large p is not proof that a dataset came from A: it says these descriptors
  cannot tell them apart.
- Choosing the split seed after seeing the result voids the guarantee; fix it
  (and `features`, `k`, `central`) up front.

## 6 · A dataset of another shape

A dataset with four treatments has no reference world of its shape:
`typicality` reports `no_reference` instead of comparing it with worlds of
another shape, and `matched_prior` refuses to generate a reference from a
configuration that never makes four active treatments.

In [ ]:
four = frames["A"].assign(radio=lambda f: (f["tv"] * 0.5 + 1.0).astype("float32"))
dataset_four = pg.ObservedWorlds.from_frame(
    four,
    treatments=["tv", "search", "social", "radio"],
    covariates=["price", "temperature"],
    outcome="sales",
)
four_result = pg.typicality(
    reference, pg.world_descriptors(dataset_four, source_id="four treatments", lags=LAGS)
)
print(four_result.table())
try:
    pg.matched_prior(priors["A"], dataset_four, n_cells=80, seed=101)
except ValueError as error:
    print("matched_prior:", error)

In [ ]:
assert four_result.to_frame()["status"].tolist() == ["no_reference"]